In [39]:
import os
import numpy as np
%run ./GetData.ipynb
import glob

In [35]:
base_dir = "/home/msp25gd/Downloads"
input_fits_path = base_dir

file_list = sorted(
    f for f in glob.glob(os.path.join(input_fits_path, "ADP.*")) if os.path.isfile(f)
 )
print(f"Matched {len(file_list)} input files")
if len(file_list) == 0:
    raise FileNotFoundError(
        f"No input files found in {input_fits_path}. Expected files like ADP.*"
    )
Spectra = GetSpectra(nb_files= len(file_list), length_spec= 2000) # 2000 points for 20 Angstrom

for i, file in enumerate(file_list):

    Spectra.get_spectrum(i, file)

    print('{}/{}'.format(i+1,len(file_list)))
    # sys.exit()
    
    if i%10000 == 0:
        Spectra.save_arr(s_path='res/spec/' , w_path='res/wavelengths/')

        
Spectra.save_arr(s_path='res/spec/' , w_path='res/wavelengths/')
print('Saved')
print('----------------------')
print(Spectra.corrupt)
print(Spectra.sK)

Matched 84188 input files
error a
error a
error b


ValueError: Length of indexer and values mismatch

In [5]:
Spectra.sK

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(84188, 2000))

In [41]:
import inspect
print(inspect.getsource(GetSpectra.get_spectrum))

    def get_spectrum(self, index, file):
        ''' Update function by creating new function that does metadata + spectra
        Only opens the file once and not for both'''
        lines = [self.CaIIH, self.CaIIK]
        print(f'processing file: {file}')

        try:
            with fits.open(file) as h:
                header = h[0].header
                wavelmin = header.get('WAVELMIN')
                wavelmax = header.get('WAVELMAX')

                if wavelmin is None or wavelmax is None:
                    raise KeyError('Missing WAVELMIN/WAVELMAX in FITS header')

                wavelength, spec, error = self._extract_spectrum_triplet(file)

                if len(wavelength) == 0:
                    raise ValueError('Empty wavelength array')

                use_nm = np.nanmedian(wavelength) < 1000
                print(f'wavelength range: WAVELMIN={wavelmin}, WAVELMAX={wavelmax}')

                for line in lines:
                    target_line = line / 10.0 if u